# Task 3 re-run (soft MoE) with the corrected validation objective
Settings: **GPU T4 x2**, **Internet On**, secret `WANDB_API_KEY`.
**Add Input -> Your Work -> Notebooks ->** select the first training notebook (its output contains `outputs/checkpoints`).
Then *Save Version -> Save & Run All*. Download `moe_results.zip` from the Output tab (~1 h).

In [ ]:
import os, glob, shutil
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as e:
    print("No W&B secret - logging offline:", e)
cands = glob.glob("/kaggle/input/**/outputs/checkpoints/t2_classifier.pt", recursive=True)
print(cands)
assert cands, "Add the first notebook's output as an input (Add Input -> Your Work)"
prev = os.path.dirname(os.path.dirname(cands[0]))
W = "/kaggle/working"
shutil.copytree(prev, f"{W}/outputs", dirs_exist_ok=True)
for f in ["checkpoints/t3_soft_moe.pt"]:
    p = f"{W}/outputs/{f}"
    if os.path.exists(p): os.remove(p)
print(os.listdir(f"{W}/outputs/checkpoints"))

In [ ]:
!git clone -q https://github.com/ibrahimmlk/Generative-AI-Assignment-1.git /kaggle/working/genai-a1
%cd /kaggle/working/genai-a1
!git log --oneline | head -3
!pip install -q optuna wandb onnx onnxscript onnxruntime

In [ ]:
!NUM_WORKERS=4 PYTHONUNBUFFERED=1 python scripts/run_restoration.py --data /kaggle/working/data --out /kaggle/working/outputs --budget full --stages moe,eval,export 2>&1 | grep -v "^wandb:" | tail -n 120

In [ ]:
%cd /kaggle/working
!zip -q -r moe_results.zip outputs/results outputs/history outputs/optuna outputs/figures outputs/onnx outputs/checkpoints/t3_soft_moe.pt
!ls -la moe_results.zip